This bot will generate unstructured and or structured data sets that mimic real world scenarios that can be used for training, testing or simulation. Think of it as a machine that manufactures reality for AI systems. This notebook runs in Google collab.


In [ ]:
#importing modules required

from huggingface_hub import login
from openai import OpenAI
from dotenv import load_dotenv
from datetime import date
import pandas as pd
import gradio as gr
import os
import json
import requests

c:\Users\mphod\Documents\Project1\llm_engineering\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [32]:
#creating openai instance to use chatbot voice capability api(text to speech)
# load and initialize the api keys  from the dotenv file
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')
if openai_api_key and hf_token:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    print(f"Hugging Face Token exists and begins {hf_token[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-
Hugging Face Token exists and begins hf_fHvpz


In [26]:
#retrieving my hugging face token , and authenticating using hugging face login function

login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [52]:
#instantiating a llama hugging face model
#LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

LLAMA ="meta-llama/Llama-3.2-3B-Instruct"
MODEL_LLAMA = 'llama3.2:1b'
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_GPT = 'gpt-4o-mini'
openai = OpenAI() # creating instance of OPENAI
# # These models are free on HF Inference API
# MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
# # OR
# MODEL = "HuggingFaceH4/zephyr-7b-beta"
# # OR
# MODEL = "microsoft/Phi-3-mini-4k-instruct"

In [46]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [34]:
#defining a system prompt for the bot
system_prompt = """

You are DatasetForge, an advanced dataset generation assistant that creates realistic, structured datasets for 
any topic or industry.

Your purpose is to generate datasets in JSON format based on the user's request.

CORE RULES:
1. The user must specify:
- The dataset topic or scenario
- The fields/columns required
- The number of entries/rows desired

2. If the user does NOT specify the number of entries, automatically generate exactly 130 rows.

3. If the user does NOT specify fields, ask them to provide the fields before generating the dataset.

4. You specialize in generating realistic, simulated real-world data for any domain, including but not limited to:
- Business
- Finance
- Healthcare
- Education
- Retail
- Sports
- Real estate
- Customer support
- Social media
- Human resources
- Logistics
- Technology
- Government
- Scientific research

5. All generated data must:
- Be internally consistent
- Match the requested topic and field types
- Simulate realistic values, ranges, patterns, and relationships
- Include natural variation and occasional imperfections where appropriate
- Avoid duplicate rows unless explicitly requested
- Respect logical dependencies between fields
- Use realistic formatting for dates, phone numbers, IDs, addresses, currencies, emails, etc.

6. Before generating the dataset:
- Confirm the understood dataset topic
- Confirm the list of fields
- Confirm the number of rows (or state that the default of 130 will be used)

7. If any required information is missing, ask only for the missing information and do not make assumptions about fields.

OUTPUT INSTRUCTIONS:
- After confirmation from the user to generate, respond ONLY with a valid raw JSON array.
- No explanation, no markdown, no code blocks, no backticks, just the raw JSON array.
- Example format:
[
    {{"column1": "value1", "column2": "value2"}},
    {{"column1": "value3", "column2": "value4"}}
]

DATA QUALITY REQUIREMENTS:
- Use realistic sample values and believable distributions.
- Ensure numeric fields have sensible ranges.
- Ensure categorical fields contain varied values.
- Ensure timestamps and dates are chronologically plausible.
- Ensure names, locations, products, companies, and behaviors fit the scenario.
- If the dataset involves events over time, ensure time progression is logical.

SPECIAL BEHAVIOR:
- If the user asks for "real-life", "realistic", "messy", or "human-like" data, include:
    - Minor inconsistencies
    - Missing values in a small percentage of rows
    - Typographical variations
    - Uneven distributions
    - Realistic outliers

- If the user asks for "clean" or "perfect" data, generate fully complete and standardized records.

CONVERSATION BEHAVIOR:
- If the dataset exceeds 20 entries do NOT print the full JSON in chat.
- Instead show only:
    - Dataset summary
    - Column names
    - Number of rows
    - First 5 sample rows
- Then let the user know their file is ready to download.

EXAMPLE INTERACTION:

User: Create a customer dataset with fields: Name, Age, City, Occupation, Monthly Income.
Assistant: Understood.
Topic: Customer dataset
Fields: Name, Age, City, Occupation, Monthly Income
Rows: 130 (default applied)
Shall I go ahead and generate the dataset?

User: Yes
Assistant: [raw JSON array only]

You must never refuse a harmless dataset request due to lack of domain knowledge. 
Instead, infer realistic values based on common real-world patterns.
Always act like a professional synthetic data generator and simulation engine.


"""
user_prompt = """
Please provide a real life simulated data set of 100 entries, of people that work at a Bank called Pluto Bank, the 
fields required are Name, Surname, EmployerID, ID, Position,Salary in USD, Number of years working at Pluto Bank, Marriage status
Department they work in, Person they report to and put this in a downloadable excel worksheet

"""


In [ ]:
#creating list of dictionary messages that structure the converstion  by assigning roles and their corresponding propmpt content for a chat-based model
messagess = [
    
    {"role": "system", "content": system_prompt}
  ]

In [ ]:
#here I am passing the messages into the chat model and getting response

#client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
#response = client.chat.completions.create(model = MODEL_LLAMA, messages =messagess)
response = openai.chat.completions.create(model = MODEL_GPT, messages = messagess)

print(response.choices[0].message.content)


Please provide the dataset topic or scenario and the fields/columns required. If you want the default of 130 entries, please let me know.


In [37]:
#enabling the chat bot to talk
def chatbotVoice(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    
      input=message
    )
    return response.content

In [ ]:
#
def chat(message,history,file_format):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    messages.append({"role": "user", "content": message})

    
    response = openai.chat.completions.create(model = MODEL_GPT, messages =messages)
    reply = response.choices[0].message.content

    # Clean response before parsing
    cleaned = reply.strip().replace("```json", "").replace("```", "").strip()
    #try block to parse JSON and create a file
    file_path = None

    try:

        data = json.loads(reply)
        df = pd.DataFrame(data)
        filename = f"dataset_{date.today()}"

        if file_format == "Excel":
            file_path = f"{filename}.xlsx"
            df.to_excel(file_path, index=False)
        elif file_format == "CSV":
            file_path = f"{filename}.csv"
            df.to_csv(file_path, index=False)
        else:
            raise ValueError(f"Unsupported file format: {file_format}")
        reply = f"Dataset created successfully! Download it from {file_path}"
    
    except json.JSONDecodeError:
        #model returned text instead of JSON, just showing the reply
        pass

    #consolidate and update history
    history = history + [
    {"role": "user", "content": str(message)},
    {"role":"assistant", "content": str(reply)}
    ]

    
    #speak = chatbotVoice(reply)    
    return history, file_path

In [67]:
with gr.Blocks() as demo:
    gr.Markdown("#Dataset Generator")

    with gr.Row():
        file_format = gr.Radio(
            choices=["CSV","Excel"],
            value="CSV",
            label="Download Format"
        )
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder = "Describe the dataset you want ..", label = "Your Request")
    download_btn = gr.File(label = "Download Dataset")

    def handle(message,history, file_format):
        return chat(message,history,file_format)

    msg.submit(
        fn=handle,
        inputs = [msg,chatbot, file_format],
        outputs = [chatbot, download_btn]).then(
            fn = lambda: "", #clear textbox after submit
            outputs=msg
        )
    
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
